In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
df_fake = pd.read_csv(r"C:\Users\Dell\Documents\Fake.csv", on_bad_lines='skip', low_memory=False)
df_true = pd.read_csv(r"C:\Users\Dell\Documents\True.csv", on_bad_lines='skip', low_memory=False)

In [3]:
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [4]:
df_fake["class"] = 0
df_true["class"] = 1

In [5]:
news = pd.concat([df_fake, df_true], ignore_index=True)

In [6]:
news = news.sample(frac = 1)
news.head()


,title,text,subject,date,class
25970,Trump to ban transgender U.S. military personn...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"July 26, 2017",1
24569,Ex-U.S. spy chiefs urge Congress to renew inte...,WASHINGTON (Reuters) - Former U.S. intelligenc...,politicsNews,"October 23, 2017",1
37029,Venezuela has not told U.N. it is changing rep...,NEW YORK (Reuters) - Venezuela s government ha...,worldnews,"November 29, 2017",1
34839,Chess federation says Israel excluded from Sau...,ATHENS (Reuters) - Israeli players have been d...,worldnews,"December 24, 2017",1
4238,WATCH: This Is Why Conservative ‘Christians’ ...,"By his own admission, Donald Trump is a serial...",News,"October 12, 2016",0


In [7]:
news.reset_index(inplace=True)
news.head()

,index,title,text,subject,date,class
0,25970,Trump to ban transgender U.S. military personn...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"July 26, 2017",1
1,24569,Ex-U.S. spy chiefs urge Congress to renew inte...,WASHINGTON (Reuters) - Former U.S. intelligenc...,politicsNews,"October 23, 2017",1
2,37029,Venezuela has not told U.N. it is changing rep...,NEW YORK (Reuters) - Venezuela s government ha...,worldnews,"November 29, 2017",1
3,34839,Chess federation says Israel excluded from Sau...,ATHENS (Reuters) - Israeli players have been d...,worldnews,"December 24, 2017",1
4,4238,WATCH: This Is Why Conservative ‘Christians’ ...,"By his own admission, Donald Trump is a serial...",News,"October 12, 2016",0


In [8]:
news = news.drop(['date','subject','index','title'], axis=1)
news.head()

,text,class
0,WASHINGTON (Reuters) - President Donald Trump ...,1
1,WASHINGTON (Reuters) - Former U.S. intelligenc...,1
2,NEW YORK (Reuters) - Venezuela s government ha...,1
3,ATHENS (Reuters) - Israeli players have been d...,1
4,"By his own admission, Donald Trump is a serial...",0


In [9]:
news.dropna(inplace=True)

In [10]:
news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    44898 non-null  object
 1   class   44898 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 701.7+ KB


In [11]:
news.shape

(44898, 2)

In [12]:
news.isnull().sum()

text     0
class    0
dtype: int64

In [13]:
news.duplicated().sum()

np.int64(6251)

In [14]:
news.drop_duplicates(inplace=True)
news.duplicated().sum()

np.int64(0)

In [16]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)
    return text.strip()

    news['text'] = news['text'].apply(clean_text)

In [17]:
def remove_source_leak(text):
    text = re.sub(r'^[A-Za-z\s,]+\(reuters\)\s*-\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters\b', '', text, flags=re.IGNORECASE)
    return text

news['text'] = news['text'].apply(remove_source_leak)
news['text'] = news['text'].apply(clean_text)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer #covert text to numerical data
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(news['text']) #feature

In [20]:
y = news['class'] #label

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 93.30%


In [22]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100)
rf_model.fit(X_train, y_train)
y_rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_rf_pred)
print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

Random Forest Accuracy: 96.53%


In [23]:
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)
y_dt_pred = dt_model.predict(X_test)
dt_accuracy = accuracy_score(y_test, y_dt_pred)
print(f"Decision Tree Accuracy: {dt_accuracy * 100:.2f}%")

Decision Tree Accuracy: 93.97%


In [24]:
from sklearn.linear_model import LogisticRegression
logistic_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logistic_model.fit(X_train, y_train)
y_logistic_pred = logistic_model.predict(X_test)
logistic_accuracy = accuracy_score(y_test, y_logistic_pred)
print(f"Logistic Regression Accuracy: {logistic_accuracy * 100:.2f}%")

Logistic Regression Accuracy: 98.05%


In [25]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_knn_pred = knn_model.predict(X_test)
knn_accuracy = accuracy_score(y_test, y_knn_pred)
print(f"KNN Accuracy: {knn_accuracy * 100:.2f}%")

KNN Accuracy: 87.34%


In [26]:
from sklearn.svm import SVC

svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)
y_svm_pred = svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, y_svm_pred)
print(f"SVM Accuracy: {svm_accuracy * 100:.2f}%")

SVM Accuracy: 98.69%


In [27]:
dt_model_fixed = DecisionTreeClassifier(
    max_depth=20,
    min_samples_leaf=5,
    min_samples_split=10,
    random_state=42
)
dt_model_fixed.fit(X_train, y_train)

train_acc = accuracy_score(y_train, dt_model_fixed.predict(X_train))
test_acc = accuracy_score(y_test, dt_model_fixed.predict(X_test))

print(f"Decision Tree (Fixed) - Train Accuracy: {train_acc*100:.2f}%")
print(f"Decision Tree (Fixed) - Test Accuracy:  {test_acc*100:.2f}%")
print(f"Gap: {(train_acc-test_acc)*100:.2f}%")

Decision Tree (Fixed) - Train Accuracy: 97.54%
Decision Tree (Fixed) - Test Accuracy:  94.35%
Gap: 3.20%


In [28]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(dt_model_fixed, X, y, cv=5, scoring='accuracy')
print(f"CV Scores: {[f'{s*100:.2f}%' for s in cv_scores]}")
print(f"Mean CV Accuracy: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")

CV Scores: ['93.36%', '93.93%', '93.67%', '94.18%', '94.75%']
Mean CV Accuracy: 93.98% (+/- 0.47%)


In [29]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_leaf': [1, 5, 10],
    'min_samples_split': [2, 10, 20]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV Accuracy: {grid_search.best_score_*100:.2f}%")

best_dt = grid_search.best_estimator_
test_acc = accuracy_score(y_test, best_dt.predict(X_test))
print(f"Test Accuracy with Best Params: {test_acc*100:.2f}%")

Best Parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV Accuracy: 94.17%
Test Accuracy with Best Params: 94.42%


In [30]:
import joblib

In [31]:
joblib.dump(logistic_model, 'model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

model = joblib.load('model.pkl')
vectorizer = joblib.load('vectorizer.pkl') 

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)
    return text.strip()

def predict_news(text):
    cleaned = clean_text(text)
    vectorized = vectorizer.transform([cleaned])
    prediction = logistic_model.predict(vectorized)[0]
    return "✅ Real News" if prediction == 1 else "❌ Fake News"

In [ ]:
new_text = input("Enter news text to predict:\n> ")
print(predict_news(new_text))